In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import os
from functools import partial
import json
import random
import torch
from datasets import Dataset, load_dataset, concatenate_datasets, interleave_datasets
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, TrainerCallback
from torch.utils.data import DataLoader
from tqdm import tqdm
from trl import SFTTrainer, SFTConfig
import wandb
import numpy as np

import sys
from pathlib import Path

PROJECT_ROOT = Path("../")
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")


In [ ]:
from src.data import (
    get_abd_dataset,
    get_ded_dataset,
    get_mth_dataset,
    get_sci_dataset, 
    get_lgc_dataset,
    load_jsonl
)

from src.config import (
    AFFINE_DATASET_REPOSITORY,
    PRIME_DATASET_REPOSITORY,
    LGC_DATASET_REPOSITORY,
    AGENTGYM_DATASET_REPOSITORY,
    TRAIN_DATA_PATH,
)

agentgym_dataset = Dataset.from_list(load_jsonl(TRAIN_DATA_PATH + "/agentgym.jsonl")).remove_columns("task_id")

swe_dataset = Dataset.from_list(load_jsonl(TRAIN_DATA_PATH + "/swe.jsonl")).remove_columns("task_id")
cde_dataset = Dataset.from_list(load_jsonl(TRAIN_DATA_PATH + "/cde.jsonl")).remove_columns("task_id")

abd_dataset = get_abd_dataset(AFFINE_DATASET_REPOSITORY).remove_columns("task_id")
ded_dataset = get_ded_dataset(AFFINE_DATASET_REPOSITORY).remove_columns("task_id")

mth_dataset = get_mth_dataset(PRIME_DATASET_REPOSITORY).remove_columns("task_id")
sci_dataset = get_sci_dataset(PRIME_DATASET_REPOSITORY).remove_columns("task_id")
lgc_dataset = get_lgc_dataset(LGC_DATASET_REPOSITORY).remove_columns("task_id")


from dotenv import load_dotenv
load_dotenv()

In [ ]:

train_dataset = lgc_dataset

train_dataset = train_dataset.rename_column(
    "conversations", "messages"
)

eval_dataset = train_dataset.shuffle(seed=42).select(range(min(50, len(train_dataset))))

print("="*60)
print(f"Train dataset size: {len(train_dataset)}")
print(f"AgentGym: {len(agentgym_dataset)}")
print(f"LGC: {len(lgc_dataset)}")
print(f"ABD: {len(abd_dataset)}")
print(f"DED: {len(ded_dataset)}")
print(f"MTH: {len(mth_dataset)}")
print(f"SCI: {len(sci_dataset)}")
print(f"SWE: {len(swe_dataset)}")
print(f"CDE: {len(cde_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")
print("="*60)


In [ ]:
# Import PEFT for LoRA
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_NAME = os.getenv("MODEL_NAME")
OUTPUT_DIR = "./sft_output_lora"

CHAT_TEMPLATE = r"""
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- for message in messages %}
    {%- if message.content is string %}
        {%- set content = message.content %}
    {%- else %}
        {%- set content = '' %}
    {%- endif %}
    {%- if (message.role == "user") or (message.role == "system" and not loop.first) %}
        {{- '<|im_start|>' + message.role + '\n' + content + '<|im_end|>' + '\n' }}
    {%- elif message.role == "assistant" %}
        {%- generation -%}
        {{- '<|im_start|>' + message.role + '\n' + content }}
        {%- if message.tool_calls %}
            {%- for tool_call in message.tool_calls %}
                {%- if (loop.first and content) or (not loop.first) %}
                    {{- '\n' }}
                {%- endif %}
                {%- if tool_call.function %}
                    {%- set tool_call = tool_call.function %}
                {%- endif %}
                {{- '<tool_call>\n{"name": "' }}
                {{- tool_call.name }}
                {{- '", "arguments": ' }}
                {%- if tool_call.arguments is string %}
                    {{- tool_call.arguments }}
                {%- else %}
                    {{- tool_call.arguments | tojson }}
                {%- endif %}
                {{- '}\n</tool_call>' }}
            {%- endfor %}
        {%- endif %}
        {{- '<|im_end|>\n' }}
        {%- endgeneration -%}
    {%- elif message.role == "tool" %}
        {%- if loop.first or (messages[loop.index0 - 1].role != "tool") %}
            {{- '<|im_start|>user' }}
        {%- endif %}
        {{- '\n<tool_response>\n' }}
        {{- content }}
        {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
{%- endif %}
""".strip()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.chat_template = CHAT_TEMPLATE

# Load model with LoRA-friendly settings
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    low_cpu_mem_usage=False,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.config.use_cache = False

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                           # LoRA rank (you can adjust: 8, 16, 32, 64)
    lora_alpha=32,                  # LoRA alpha (typically 2*r)
    lora_dropout=0.1,               # Dropout probability
    target_modules=[                # Target modules to apply LoRA
        "q_proj",
        "k_proj", 
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    inference_mode=False,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

# If we added a token, resize embeddings
if model.config.vocab_size != len(tokenizer):
    model.resize_token_embeddings(len(tokenizer))



In [ ]:
from src.config import get_sampling_range

sampling_range = get_sampling_range()

env_keymap = {
    "MTH": "mth",
    "SCI": "sci",
    "LGC": "lgc",
    "CDE": "cde",
    "agentgym:webshop": "webshop",
    "agentgym:alfworld": "alfworld",
    "agentgym:sciworld": "sciworld",
    "agentgym:textcraft": "textcraft",  
    "affine:abd-v2": "abd-v2",
    "affine:ded-v2": "ded-v2",
    "SWE-PRO": "swe-pro",
}

eval_ids = {env_keymap[k]: random.sample(v['sampling_config']['sampling_list'], k=1) for k, v in sampling_range.items()}

In [ ]:
eval_ids.pop('swe-pro', None)

In [ ]:
from src.evaluation import evaluate_env

In [ ]:
tokenizer.decode([151645])

In [ ]:
from src.evaluation import evaluate_env
from transformers import TrainerCallback

class MetricsCallback(TrainerCallback):
    def __init__(self, model, tokenizer, eval_ids, eval_steps=8):
        super().__init__()
        self.model = model
        self.tokenizer = tokenizer
        self.eval_ids = eval_ids
        self.eval_steps = eval_steps
        
    async def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.eval_steps == 0:
            model.eval()
            wandb.log({"step": state.global_step})
            scores = []
            with torch.no_grad():
                for env, task_ids in eval_ids.items():
                    result = await evaluate_env(model, env=env, task_ids=task_ids, tokenizer=tokenizer)
                    score = result.get("score", 0.0)
                    wandb.log({f"{env}_score": score}, step=state.global_step)
                    scores.append(score)
            wandb.log({"mean_score": sum(scores) / len(scores)}, step=state.global_step)
            model.train()
            

In [ ]:
import torch
import wandb
from transformers import TrainerCallback

class MetricsCallback(TrainerCallback):
    def __init__(self, eval_steps, model, tokenizer, eval_ids):
        super().__init__()
        self.eval_steps = eval_steps
        self.model = model
        self.tokenizer = tokenizer
        self.eval_ids = eval_ids

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step == 0:
            return control
        
        if state.global_step % self.eval_steps == 0:
            # Run evaluation synchronously
            self._run_eval(state.global_step)
        
        return control
    
    def _run_eval(self, step):
        """Synchronous wrapper that runs async evaluation"""
        import asyncio
        asyncio.run(self._async_eval(step))
    
    async def _async_eval(self, step):
        self.model.eval()
        scores = []
        with torch.no_grad():
            for env, task_ids in self.eval_ids.items():
                result = await evaluate_env(
                    self.model,
                    env=env,
                    task_ids=task_ids,
                    tokenizer=self.tokenizer,
                )
                score = result.get("score", 0.0)
                wandb.log({f"{env}_score": score}, step=step)
                scores.append(score)
        
        if scores:
            wandb.log({"mean_score": sum(scores) / len(scores)}, step=step)
        
        self.model.train()

In [ ]:
with torch.no_grad():
    for env, task_ids in eval_ids.items():
        score = await evaluate_env(model, env=env, task_ids=task_ids, tokenizer=tokenizer)
        print(f"{env}: {score['score']}")

In [ ]:
tokenizer.chat_template = CHAT_TEMPLATE

In [ ]:
eval_dataset = train_dataset.shuffle(seed=42).select(range(min(16, len(train_dataset))))
train_dataset = train_dataset.shuffle(seed=42).select(range(min(128, len(train_dataset))))


In [ ]:

NUM_PROC = 8

training_args = SFTConfig(
    model_init_kwargs={"torch_dtype": torch.bfloat16},
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=15,
    learning_rate=1e-05,
    warmup_steps=32,
    max_length=8192,
    max_grad_norm=1.0,
    dataset_num_proc=8,
    
    # Conversation control
    assistant_only_loss=True,
            
    # Logging
    logging_strategy="steps",
    logging_steps=8,
    
    # Evaluation
    do_eval=True,
    eval_strategy="steps",
    eval_steps=8,
    save_total_limit=3,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Checkpointing
    output_dir=OUTPUT_DIR,
    save_strategy="steps",
    save_steps=8,
    bf16=True,
    remove_unused_columns=False,
    report_to=["wandb"],
    
    # Memory optimization
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)


# Initialize wandb
wandb.init(
    project="SFT - lora",
    name="LGC improvement",
    config={
        "model_name": MODEL_NAME,
        "train_dataset": "LGC",
        "lora_r": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
        "training_method": "LoRA",
        **training_args.to_dict(),
    }
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    callbacks=[MetricsCallback(eval_steps=8, model=model, tokenizer=tokenizer, eval_ids=eval_ids)],
)

In [ ]:
eval_ids = {env_keymap[k]: random.sample(v['sampling_config']['sampling_list'], k=5) for k, v in sampling_range.items()}

In [ ]:

trainer.train()

# Save the LoRA adapter
model.save_pretrained(OUTPUT_DIR + "/final_lora_adapter")
tokenizer.save_pretrained(OUTPUT_DIR + "/final_lora_adapter")

wandb.finish()

print("Training complete! LoRA adapter saved to:", OUTPUT_DIR + "/final_lora_adapter")

In [ ]:
!pip install bitsandbytes